In [ ]:
import os
from typing import Dict, List, Optional

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt


# =========================
# Settings
# =========================
BASE_DIR = "."  # graph_*.gexf 가 있는 폴더에서 실행
OUT_BASE = os.path.join(BASE_DIR, "layouts_all")

FIGSIZE = (16, 12)
DPI = 240
NODE_SIZE = 700
ARROWSIZE = 14
FONT_SIZE = 9

TOPK_EDGE_LABELS = 30

# ✅ 10가지 시각화(레이아웃) 목록
# - 설치 없이 동작하는 networkx 레이아웃 7개
# - Graphviz 3개(있으면 사용, 없으면 spring으로 fallback)
LAYOUTS = [
    "spring",
    "kamada_kawai",
    "spectral",
    "circular",
    "shell",
    "random",
    "spiral",
    "multipartite_topo",   # DAG depth 기반
    "graphviz_dot",        # (옵션) Graphviz
    "graphviz_sfdp",       # (옵션) Graphviz
]

# Graphviz가 없을 때 fallback
FALLBACK_LAYOUT = "spring"


# =========================
# Utilities
# =========================
def ensure_dir(p: str) -> None:
    os.makedirs(p, exist_ok=True)


def load_gexf_as_digraph(path: str) -> nx.DiGraph:
    """
    - MultiDiGraph 방지
    - 노드명 문자열 통일
    - ✅ multi edge를 "합산"하지 않음 (스케일 튀는 문제 방지)
      -> 동일 (u,v) 중 |score| 또는 |weight|가 큰 것만 유지
    """
    G0 = nx.read_gexf(path)
    G = nx.DiGraph()

    for n, data in G0.nodes(data=True):
        G.add_node(str(n), **data)

    def as_float(x) -> float:
        try:
            return float(x)
        except Exception:
            return 0.0

    if isinstance(G0, nx.MultiDiGraph):
        for u, v, k, data in G0.edges(keys=True, data=True):
            u2, v2 = str(u), str(v)

            # normalize numeric attrs
            nd = dict(data)
            for attr in ["weight", "score", "w"]:
                if attr in nd:
                    nd[attr] = as_float(nd.get(attr, 0.0))

            if not G.has_edge(u2, v2):
                G.add_edge(u2, v2, **nd)
            else:
                # ✅ 합산 금지: |attr| 큰 것으로 교체
                for attr in ["score", "weight", "w"]:
                    if attr in nd:
                        oldv = as_float(G[u2][v2].get(attr, 0.0))
                        newv = as_float(nd.get(attr, 0.0))
                        if abs(newv) > abs(oldv):
                            G[u2][v2][attr] = newv
    else:
        for u, v, data in G0.edges(data=True):
            u2, v2 = str(u), str(v)
            nd = dict(data)
            for attr in ["weight", "score", "w"]:
                if attr in nd:
                    nd[attr] = as_float(nd.get(attr, 0.0))
            G.add_edge(u2, v2, **nd)

    G.remove_edges_from(list(nx.selfloop_edges(G)))
    return G


def normalize_pos(G: nx.DiGraph, pos: Dict) -> Dict:
    pos2 = {str(k): (float(v[0]), float(v[1])) for k, v in (pos or {}).items()}
    missing = [n for n in G.nodes() if n not in pos2]
    if missing:
        fallback = nx.spring_layout(G.to_undirected(), seed=42)
        for n in missing:
            xy = fallback.get(n, (0.0, 0.0))
            pos2[n] = (float(xy[0]), float(xy[1]))
    return pos2


def compute_edge_widths(values: List[float], base_width: float = 1.5) -> List[float]:
    """|value|를 [0.6, 4.0]으로 매핑"""
    if not values:
        return []
    absvals = [abs(v) for v in values]
    vmin, vmax = min(absvals), max(absvals)
    if vmax - vmin < 1e-12:
        return [base_width for _ in values]
    return [0.6 + 3.4 * (a - vmin) / (vmax - vmin) for a in absvals]


def topo_levels(G: nx.DiGraph) -> Dict[str, int]:
    """
    DAG depth 기반 레벨(토폴로지 depth).
    DAG가 아니면 in-degree로 fallback.
    """
    try:
        order = list(nx.topological_sort(G))
    except Exception:
        return {n: int(G.in_degree(n)) for n in G.nodes()}

    level = {n: 0 for n in G.nodes()}
    for v in order:
        preds = list(G.predecessors(v))
        if preds:
            level[v] = 1 + max(level[p] for p in preds)
    return level


def pick_layout(G: nx.DiGraph, layout_name: str) -> Dict:
    UG = G.to_undirected()

    if layout_name == "spring":
        return nx.spring_layout(UG, seed=42)
    if layout_name == "kamada_kawai":
        return nx.kamada_kawai_layout(UG)
    if layout_name == "spectral":
        return nx.spectral_layout(UG)
    if layout_name == "circular":
        return nx.circular_layout(UG)
    if layout_name == "shell":
        return nx.shell_layout(UG)
    if layout_name == "random":
        return nx.random_layout(UG, seed=42)
    if layout_name == "spiral":
        return nx.spiral_layout(UG)

    if layout_name == "multipartite_topo":
        levels = topo_levels(G)
        for n, lv in levels.items():
            G.nodes[n]["subset"] = int(lv)
        # subset_key="subset"이 있어야 레이어가 의미있게 나옴
        return nx.multipartite_layout(G, subset_key="subset")

    # Graphviz layouts (옵션: 설치되어 있으면)
    if layout_name.startswith("graphviz_"):
        prog = layout_name.replace("graphviz_", "")
        try:
            from networkx.drawing.nx_pydot import graphviz_layout
            return graphviz_layout(G, prog=prog)
        except Exception:
            # fallback
            return pick_layout(G, FALLBACK_LAYOUT)

    # default fallback
    return pick_layout(G, FALLBACK_LAYOUT)


def pick_edge_attr_for_alg(alg: str, G: nx.DiGraph) -> Optional[str]:
    if alg == "PC":
        return None
    candidates = ["score", "weight", "w"] if alg == "GES" else ["weight", "score", "w"]
    edges = list(G.edges())
    if not edges:
        return None
    for attr in candidates:
        for u, v in edges[: min(50, len(edges))]:
            if attr in G[u][v]:
                return attr
    return None


# =========================
# Drawing
# =========================
def draw_graph(
    alg: str,
    G: nx.DiGraph,
    edge_attr: Optional[str],
    layout_name: str,
    out_dir: str
) -> None:
    ensure_dir(out_dir)
    out_png = os.path.join(out_dir, f"{alg}__{layout_name}.png")

    pos = normalize_pos(G, pick_layout(G, layout_name))
    edges = list(G.edges())

    # edge values
    if edge_attr is None:
        values = [0.0 for _ in edges]
    else:
        values = []
        for u, v in edges:
            val = G[u][v].get(edge_attr, 0.0)
            try:
                val = float(val)
            except Exception:
                val = 0.0
            values.append(val)

    widths = compute_edge_widths(values, base_width=1.5)

    # colors
    if edge_attr is None:
        edge_colors = ["0.35" for _ in edges]
    else:
        edge_colors = ["tab:blue" if v > 0 else "tab:red" if v < 0 else "0.5" for v in values]

    # edge labels (top-k by |value|)
    edge_labels = {}
    if TOPK_EDGE_LABELS > 0 and edge_attr is not None and len(edges) > 0:
        edges_sorted = sorted(
            edges,
            key=lambda e: abs(float(G[e[0]][e[1]].get(edge_attr, 0.0) or 0.0)),
            reverse=True
        )
        top_edges = edges_sorted[: min(TOPK_EDGE_LABELS, len(edges_sorted))]
        for u, v in top_edges:
            edge_labels[(u, v)] = f"{float(G[u][v].get(edge_attr, 0.0) or 0.0):.2f}"

    plt.figure(figsize=FIGSIZE, dpi=DPI)
    title_suffix = ("structure only" if edge_attr is None else edge_attr)
    plt.title(f"{alg} | {layout_name} | {title_suffix}")

    nx.draw_networkx_nodes(G, pos, node_size=NODE_SIZE, node_color="lightgray")
    nx.draw_networkx_labels(G, pos, font_size=FONT_SIZE)

    nx.draw_networkx_edges(
        G, pos,
        edge_color=edge_colors,
        width=widths if widths else 1.5,
        arrows=True,
        arrowsize=ARROWSIZE,
        alpha=0.85,
        connectionstyle="arc3,rad=0.05",
    )

    if edge_labels:
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=FONT_SIZE)

    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()

    print(f"[DONE] {alg} | {layout_name} -> {out_png}")


# =========================
# Main
# =========================
def main():
    ensure_dir(OUT_BASE)

    paths = {
        "NOTEARS": os.path.join(BASE_DIR, "graph_NOTEARS.gexf"),
        "GOLEM": os.path.join(BASE_DIR, "graph_GOLEM.gexf"),
        "PC": os.path.join(BASE_DIR, "graph_PC.gexf"),
        "GES": os.path.join(BASE_DIR, "graph_GES.gexf"),
    }

    graphs: Dict[str, nx.DiGraph] = {}
    for alg, p in paths.items():
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing: {p}")
        graphs[alg] = load_gexf_as_digraph(p)

    for alg, G in graphs.items():
        edge_attr = pick_edge_attr_for_alg(alg, G)
        out_dir = os.path.join(OUT_BASE, alg)

        for layout in LAYOUTS:
            draw_graph(
                alg=alg,
                G=G,
                edge_attr=edge_attr,
                layout_name=layout,
                out_dir=out_dir
            )

    print(f"[ALL DONE] outputs -> {OUT_BASE}")
    print("Layouts used:", ", ".join(LAYOUTS))


if __name__ == "__main__":
    main()
